<a href="https://colab.research.google.com/github/jahnavib529/Deep-learning/blob/main/DL_Assignment_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import RegularPolygon, Circle
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
import json, time, os

np.random.seed(42)
tf.random.set_seed(42)

OUT = "/home/claude/demo/out"
os.makedirs(OUT, exist_ok=True)

IMG_SIZE = 32
CLASSES = ["Stop", "Speed Limit", "Yield", "No Entry", "Warning", "Mandatory"]
NUM_CLASSES = len(CLASSES)

def draw_sign(cls_idx, jitter=True):
    fig = plt.figure(figsize=(IMG_SIZE/100, IMG_SIZE/100), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    ax.set_facecolor((0.85, 0.87, 0.9) if not jitter else
                      (0.8+np.random.rand()*0.15, 0.8+np.random.rand()*0.15, 0.85+np.random.rand()*0.1))
    cx, cy = 0.5, 0.5
    if jitter:
        cx += np.random.uniform(-0.08, 0.08)
        cy += np.random.uniform(-0.08, 0.08)
    size = 0.38 + (np.random.uniform(-0.08, 0.08) if jitter else 0)
    rot = np.random.uniform(-25, 25) if jitter else 0

    shape_color = {
        0: ("octagon", "#d62728"),   # Stop
        1: ("circle", "#ffffff"),    # Speed Limit (white w/ red ring)
        2: ("triangle_down", "#ffcc00"),  # Yield
        3: ("circle", "#d62728"),    # No Entry
        4: ("triangle_up", "#ffcc00"),    # Warning
        5: ("circle", "#1f5fd6"),    # Mandatory
    }[cls_idx]
    shape, color = shape_color
    if jitter:
        r, g, b = matplotlib.colors.to_rgb(color)
        noise = np.random.uniform(-0.08, 0.08, 3)
        color = tuple(np.clip([r+noise[0], g+noise[1], b+noise[2]], 0, 1))

    if shape == "circle":
        ax.add_patch(Circle((cx, cy), size, facecolor=color, edgecolor="#d62728", linewidth=3))
    elif shape == "octagon":
        ax.add_patch(RegularPolygon((cx, cy), numVertices=8, radius=size, orientation=np.radians(rot),
                                     facecolor=color, edgecolor="white", linewidth=2))
    elif shape == "triangle_down":
        ax.add_patch(RegularPolygon((cx, cy), numVertices=3, radius=size, orientation=np.radians(180+rot),
                                     facecolor=color, edgecolor="#333333", linewidth=2))
    elif shape == "triangle_up":
        ax.add_patch(RegularPolygon((cx, cy), numVertices=3, radius=size, orientation=np.radians(rot),
                                     facecolor=color, edgecolor="#333333", linewidth=2))

    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    w, h = fig.canvas.get_width_height()
    img = buf.reshape(h, w, 4)[:, :, :3].copy()
    plt.close(fig)
    if jitter:
        img = img.astype(np.float32) + np.random.normal(0, 10, img.shape)
        if np.random.rand() < 0.3:
            oh, ow = np.random.randint(4, 9), np.random.randint(4, 9)
            oy, ox = np.random.randint(0, IMG_SIZE - oh), np.random.randint(0, IMG_SIZE - ow)
            img[oy:oy+oh, ox:ox+ow] = np.random.randint(150, 220)
        img = np.clip(img, 0, 255).astype(np.uint8)
    return img

def build_dataset(n_per_class, jitter=True):
    X, y = [], []
    for c in range(NUM_CLASSES):
        for _ in range(n_per_class):
            X.append(draw_sign(c, jitter=jitter))
            y.append(c)
    X = np.array(X, dtype=np.float32) / 255.0
    y = np.array(y, dtype=np.int64)
    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

print("Generating synthetic dataset...")
X_train, y_train = build_dataset(280, jitter=True)
X_val, y_val = build_dataset(60, jitter=True)
X_test, y_test = build_dataset(60, jitter=True)
print(f"train={X_train.shape}, val={X_val.shape}, test={X_test.shape}")

def build_cnn(num_classes=NUM_CLASSES, input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D(2),

        layers.Conv2D(128, 3, padding='same'),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.MaxPooling2D(2),

        layers.Flatten(),
        layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_cnn()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
total_params = model.count_params()
print("Total trainable params:", total_params)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5)
]

t0 = time.time()
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                     epochs=25, batch_size=32, callbacks=callbacks, verbose=2)
train_time = time.time() - t0

t0 = time.time()
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
infer_time_per_img = (time.time() - t0) / len(X_test)

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_conf = np.max(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=CLASSES, output_dict=True, zero_division=0)
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
w_precision, w_recall, w_f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)

# --- Plot 1: training curves ---
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(history.history['loss'], label='train loss')
axes[0].plot(history.history['val_loss'], label='val loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss')
axes[0].legend()
axes[1].plot(history.history['accuracy'], label='train acc')
axes[1].plot(history.history['val_accuracy'], label='val acc')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Accuracy')
axes[1].legend()
fig.suptitle('Training curves (synthetic dry-run data)')
fig.tight_layout()
fig.savefig(f'{OUT}/curves.png', dpi=150)
plt.close(fig)

# --- Plot 2: confusion matrix ---
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES, fontsize=7)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (synthetic dry-run)', fontsize=9)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=7)
fig.tight_layout()
fig.savefig(f'{OUT}/confusion_matrix.png', dpi=150)
plt.close(fig)

# --- Sample predictions grid ---
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
sample_idx = np.random.choice(len(X_test), 10, replace=False)
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(X_test[idx]); ax.axis('off')
    correct = y_pred[idx] == y_test[idx]
    ax.set_title(f"T:{CLASSES[y_test[idx]][:4]}\nP:{CLASSES[y_pred[idx]][:4]} ({y_conf[idx]:.2f})",
                 fontsize=7, color='green' if correct else 'red')
fig.suptitle('Sample predictions (synthetic dry-run)', fontsize=10)
fig.tight_layout()
fig.savefig(f'{OUT}/sample_predictions.png', dpi=150)
plt.close(fig)

# --- Test case table data (first 6 test images) ---
test_cases = []
for i in range(min(6, len(X_test))):
    test_cases.append({
        "id": f"TC{i+1:02d}",
        "expected": CLASSES[y_test[i]],
        "predicted": CLASSES[y_pred[i]],
        "confidence": float(y_conf[i]),
        "pass": bool(y_pred[i] == y_test[i])
    })

summary = {
    "num_classes": NUM_CLASSES,
    "classes": CLASSES,
    "train_size": len(X_train), "val_size": len(X_val), "test_size": len(X_test),
    "total_params": int(total_params),
    "epochs_run": len(history.history['loss']),
    "train_time_sec": round(train_time, 2),
    "inference_time_per_image_ms": round(infer_time_per_img * 1000, 3),
    "test_loss": round(float(test_loss), 4),
    "test_accuracy": round(float(test_acc), 4),
    "macro_precision": round(float(precision), 4),
    "macro_recall": round(float(recall), 4),
    "macro_f1": round(float(f1), 4),
    "weighted_f1": round(float(w_f1), 4),
    "test_cases": test_cases,
    "class_report": report
}
with open(f'{OUT}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

Generating synthetic dataset...
train=(1680, 32, 32, 3), val=(360, 32, 32, 3), test=(360, 32, 32, 3)
Total trainable params: 357190
Epoch 1/25
53/53 - 9s - 168ms/step - accuracy: 0.8655 - loss: 0.4476 - val_accuracy: 0.4694 - val_loss: 1.4715 - learning_rate: 0.0010
Epoch 2/25
53/53 - 5s - 103ms/step - accuracy: 0.9476 - loss: 0.1670 - val_accuracy: 0.4639 - val_loss: 1.4386 - learning_rate: 0.0010
Epoch 3/25
53/53 - 5s - 102ms/step - accuracy: 0.9565 - loss: 0.1227 - val_accuracy: 0.3139 - val_loss: 1.7543 - learning_rate: 0.0010
Epoch 4/25
53/53 - 13s - 247ms/step - accuracy: 0.9601 - loss: 0.1159 - val_accuracy: 0.4556 - val_loss: 1.6168 - learning_rate: 0.0010
Epoch 5/25
53/53 - 8s - 143ms/step - accuracy: 0.9643 - loss: 0.1029 - val_accuracy: 0.6306 - val_loss: 0.7690 - learning_rate: 0.0010
Epoch 6/25
53/53 - 11s - 209ms/step - accuracy: 0.9655 - loss: 0.1022 - val_accuracy: 0.8500 - val_loss: 0.3277 - learning_rate: 0.0010
Epoch 7/25
53/53 - 11s - 205ms/step - accuracy: 0.9726 -